# Módulo 9 · Escalado: Big Data, Entrenamiento Distribuido y MLOps

## Objetivos
1. Elegir la herramienta de datos adecuada por escala: **pandas → Polars/DuckDB (una máquina) → Spark (clúster)**, y entender las **arquitecturas de datos** (data warehouse, lake, **lakehouse**, mesh) sobre las que vive el ML.
2. Entender el **entrenamiento distribuido** moderno: paralelismo de datos (**DDP**), *sharding* de parámetros (**FSDP / ZeRO**), paralelismo de tensores y pipeline; **Horovod está deprecado**.
3. Aplicar las técnicas que hacen entrenable un modelo grande: acumulación de gradientes, precisión mixta, *checkpointing* de activaciones, `torch.compile`.
4. Diseñar el ciclo **MLOps**: versionado de datos/modelos, *experiment tracking* (MLflow), *feature store*, registro de modelos, despliegue (batch/online/streaming) y **monitorización de deriva**.
5. Implementar detección de **data drift** (PSI, KS) y razonar sobre *continual learning* y pruebas en producción.
6. Extender el ciclo a LLMs (**LLMOps**): evaluación continua, versionado de prompts, observabilidad de trazas.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Big Data y tipos de arquitecturas de datos | Serra, *Deciphering Data Architectures* (2024) | **Cap. 1** Big Data · **Cap. 2** Types of Data Architectures · **Cap. 3** The Architecture Design Session |
| Arquitecturas concretas | Serra | **Cap. 10** The Modern Data Warehouse · **Cap. 11** Data Fabric · **Cap. 12** Data Lakehouse · **Cap. 13-14** Data Mesh |
| Fundamentos de ingeniería de datos para ML | Huyen, *Designing ML Systems* (2022) | **Cap. 3** Data Engineering Fundamentals · **Cap. 4** Training Data · **Cap. 5** Feature Engineering |
| Desarrollo de modelos, entrenamiento distribuido, AutoML, evaluación offline | Huyen, DMLS | **Cap. 6** Model Development and Offline Evaluation |
| Despliegue y servicio de predicciones | Huyen, DMLS | **Cap. 7** Model Deployment and Prediction Service |
| Deriva de datos y monitorización | Huyen, DMLS | **Cap. 8** Data Distribution Shifts and Monitoring |
| Continual learning y pruebas en producción | Huyen, DMLS | **Cap. 9** Continual Learning and Test in Production |
| Infraestructura y herramientas MLOps | Huyen, DMLS | **Cap. 10** Infrastructure and Tooling for MLOps |
| Entrenamiento y despliegue a escala con TF/Keras, GPUs, Vertex AI | Géron, *Hands-On ML* (3.ª ed.) | **Cap. 19** Training and Deploying TensorFlow Models at Scale |
| Rendimiento computacional, multi-GPU, servidores de parámetros | Zhang et al., *D2L* | **Cap. 13** Computational Performance |
| Inferencia optimizada y arquitectura de aplicaciones LLM | Huyen, *AI Engineering* (2025) | **Cap. 9** Inference Optimization · **Cap. 10** AI Engineering Architecture |

> **Nota de actualización (2026).** La malla original citaba "Huyen Cap. 6 (Diseño de sistemas en producción)" y "Serra Cap. 4": Huyen Cap. 6 es *Model Development*; el diseño para producción está en Cap. 7-10. En Serra, las arquitecturas concretas están en Cap. 10-14. Se **elimina Horovod** (deprecado; Databricks lo retiró tras 15.4 LTS) en favor de **PyTorch DDP/FSDP, DeepSpeed, Ray Train y Hugging Face Accelerate**; se añaden **Polars/DuckDB**, el formato **Lakehouse (Delta/Iceberg)** y LLMOps.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 9 · Escalado, Big Data y MLOps**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, json, os, tempfile
rng = np.random.default_rng(0)

## 1. Datos: elige la herramienta por escala (Serra Cap. 1-2; Huyen DMLS Cap. 3)

| Tamaño | Herramienta 2026 | Por qué |
|---|---|---|
| < 1-5 GB | **pandas 2.x** (backend Arrow) | ubicuo, ecosistema |
| 5-500 GB en una máquina | **Polars** (Rust, multihilo, *lazy*) o **DuckDB** (SQL analítico *in-process*) | 5-50× más rápido que pandas, sin clúster |
| > TBs, muchos usuarios | **Apache Spark 4** (PySpark, Spark Connect), o motores del lakehouse (Databricks, Snowflake, BigQuery, Trino) | paralelismo en clúster, gobernanza |
| Streaming | Kafka + Flink / Spark Structured Streaming | latencia de segundos |

**Formato**: columnar **Parquet** para analítica; **Arrow** como formato en memoria compartido entre pandas/Polars/DuckDB/Spark (cero copias). **Tablas transaccionales** sobre el lago (Delta Lake, Apache Iceberg, Hudi) = **lakehouse** (Serra Cap. 12): ACID, *time travel*, evolución de esquema — el sustrato estándar del ML en 2026.

Medimos pandas vs Polars/DuckDB (si están instalados) en una agregación típica de *feature engineering*.

In [ ]:
n = 2_000_000
df = pd.DataFrame({"cliente": rng.integers(0, 50_000, n), "categoria": rng.choice(["A", "B", "C", "D"], n),
                   "monto": rng.exponential(50, n).round(2), "ts": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 180 * 86400, n), unit="s")})
tmp = tempfile.mkdtemp(); ruta = os.path.join(tmp, "transacciones.parquet"); df.to_parquet(ruta, index=False)
print(f"{n:,} filas -> Parquet {os.path.getsize(ruta)/1e6:.1f} MB")

def feats_pandas():
    d = pd.read_parquet(ruta)
    return (d.assign(mes=d["ts"].dt.to_period("M")).groupby(["cliente", "mes"], observed=True)
             .agg(n_tx=("monto", "size"), total=("monto", "sum"), max_tx=("monto", "max")).reset_index())
t0 = time.time(); f_pd = feats_pandas(); print(f"pandas : {time.time()-t0:.2f}s  ({len(f_pd):,} filas de features)")

try:
    import polars as pl
    def feats_polars():
        return (pl.scan_parquet(ruta).with_columns(pl.col("ts").dt.truncate("1mo").alias("mes"))
                  .group_by("cliente", "mes").agg(pl.len().alias("n_tx"), pl.col("monto").sum().alias("total"), pl.col("monto").max().alias("max_tx"))
                  .collect())
    t0 = time.time(); f_pl = feats_polars(); print(f"polars : {time.time()-t0:.2f}s  (lazy + multihilo)")
except ImportError: print("polars no instalado (uv pip install polars)")
try:
    import duckdb
    t0 = time.time()
    f_db = duckdb.sql(f"SELECT cliente, date_trunc('month', ts) AS mes, count(*) n_tx, sum(monto) total, max(monto) max_tx FROM '{ruta}' GROUP BY 1,2").df()
    print(f"duckdb : {time.time()-t0:.2f}s  (SQL directo sobre Parquet)")
except ImportError: print("duckdb no instalado (uv pip install duckdb)")

### PySpark: el mismo cálculo en un clúster (OPCIONAL)
```python
# uv pip install pyspark  (requiere Java 17+). Spark 4 (2025) añade Spark Connect, ANSI SQL por defecto, tipo VARIANT.
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("features").getOrCreate()
tx = spark.read.parquet(ruta)
feats = (tx.withColumn("mes", F.date_trunc("month", "ts")).groupBy("cliente", "mes")
           .agg(F.count("*").alias("n_tx"), F.sum("monto").alias("total"), F.max("monto").alias("max_tx")))
feats.write.format("delta").mode("overwrite").saveAsTable("features.cliente_mes")     # tabla lakehouse (Delta Lake)
# ML distribuido: pyspark.ml (MLlib) para modelos clásicos; para DL usa TorchDistributor (Spark 3.4+) o Ray on Spark.
```
**Regla de Serra (Cap. 3):** no elijas la arquitectura por moda; parte de casos de uso, volumen, latencia, gobernanza y equipo. Un lakehouse bien gobernado cubre el 80 % de los casos; el *data mesh* es un modelo **organizativo** (dominios dueños de sus datos), no una tecnología.

## 2. Entrenamiento distribuido (D2L Cap. 13; Huyen DMLS Cap. 6; Géron Cap. 19)

**Paralelismo de datos (DDP):** cada GPU tiene una copia completa del modelo y procesa un *shard* del lote; los gradientes se promedian con **all-reduce** (NCCL). Escala casi linealmente mientras el modelo quepa en una GPU.

**Sharding de estado (FSDP / DeepSpeed ZeRO-3):** parámetros, gradientes y estados del optimizador se **reparten** entre GPUs y se reúnen solo cuando se necesitan (all-gather por capa). Permite entrenar modelos que no caben en una GPU. Memoria por parámetro en entrenamiento con Adam en precisión mixta: ≈ **16 bytes** (2 pesos bf16 + 4 pesos fp32 maestros + 4+4 momentos + 2 grad) → un 7B necesita ~112 GB *solo* de estados: por eso se *shardea*.

**Paralelismo de tensores (TP)** parte cada matriz entre GPUs (dentro de un nodo, NVLink); **pipeline (PP)** parte por capas; **paralelismo de secuencia/contexto** para contextos largos. Los LLMs se entrenan con **3D/4D parallelism** (Megatron-LM, DeepSpeed, torchtitan).

Simulamos DDP y *gradient accumulation* en CPU para ver que son **matemáticamente equivalentes** a un lote grande.

In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F; TORCH = True; torch.manual_seed(0)
except ImportError:
    TORCH = False; print("PyTorch no instalado")

if TORCH:
    modelo = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
    X = torch.randn(256, 20); y = torch.randn(256, 1)

    # (a) lote grande de 256 en un solo dispositivo
    modelo.zero_grad(); F.mse_loss(modelo(X), y).backward(); g_ref = torch.cat([p.grad.flatten() for p in modelo.parameters()]).clone()

    # (b) 'DDP' simulado: 4 réplicas con 64 muestras cada una, all-reduce = promedio de gradientes
    grads = []
    for k in range(4):
        modelo.zero_grad(); F.mse_loss(modelo(X[k*64:(k+1)*64]), y[k*64:(k+1)*64]).backward()
        grads.append(torch.cat([p.grad.flatten() for p in modelo.parameters()]).clone())
    g_ddp = torch.stack(grads).mean(0)

    # (c) acumulación de gradientes: 4 micro-lotes secuenciales, pérdida dividida por el número de pasos
    modelo.zero_grad()
    for k in range(4): (F.mse_loss(modelo(X[k*64:(k+1)*64]), y[k*64:(k+1)*64]) / 4).backward()
    g_acc = torch.cat([p.grad.flatten() for p in modelo.parameters()]).clone()
    print("DDP (promedio) == lote grande:", torch.allclose(g_ref, g_ddp, atol=1e-6), "| acumulación == lote grande:", torch.allclose(g_ref, g_acc, atol=1e-6))

    # Presupuesto de memoria de entrenamiento (regla de los 16 bytes/parámetro + activaciones)
    for nombre, params in [("ResNet-50", 25.6e6), ("Llama-3-8B", 8.0e9), ("Llama-3-70B", 70e9)]:
        gb = params * 16 / 1e9; print(f"{nombre:12s} estados de entrenamiento (Adam, mixta): {gb:8.1f} GB -> GPUs de 80 GB necesarias (solo estados): {int(np.ceil(gb/80))}")

### El código real (PyTorch ≥ 2.4) — OPCIONAL, requiere varias GPUs
```python
# torchrun --nproc_per_node=8 train.py
import torch.distributed as dist
from torch.distributed.fsdp import fully_shard          # FSDP2 (PyTorch 2.4+): API por módulo, composable
dist.init_process_group("nccl"); rank = dist.get_rank(); torch.cuda.set_device(rank)
model = MiTransformer().cuda()
for bloque in model.bloques: fully_shard(bloque)         # shardea cada bloque; reúne parámetros solo durante su forward/backward
fully_shard(model)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, fused=True)
sampler = torch.utils.data.DistributedSampler(dataset)   # cada rank ve una partición distinta
for xb, yb in DataLoader(dataset, sampler=sampler, batch_size=32, num_workers=4, pin_memory=True):
    with torch.autocast("cuda", dtype=torch.bfloat16):
        loss = F.cross_entropy(model(xb.cuda()), yb.cuda())
    loss.backward(); opt.step(); opt.zero_grad()
# Checkpoints distribuidos: torch.distributed.checkpoint (DCP).
```
Capas de abstracción que evitan escribir esto a mano: **Hugging Face Accelerate** (`accelerate launch`), **PyTorch Lightning / Fabric**, **Ray Train** (orquesta multi-nodo, tolerancia a fallos), **DeepSpeed** (ZeRO, *offloading* a CPU/NVMe). En la nube: SageMaker, Vertex AI, Azure ML; en Databricks: `TorchDistributor`. **Horovod** (Uber, 2017) fue el estándar de la era TF1; hoy está en mantenimiento y no se recomienda para proyectos nuevos.

*Activation checkpointing* (`torch.utils.checkpoint`) cambia cómputo por memoria (recalcula activaciones en backward, ~30 % más lento, hasta 5× menos memoria). *Mixed precision* bf16 ≈ 2× de velocidad en tensor cores.

## 3. MLOps: del notebook al sistema (Huyen DMLS Cap. 7, 10)

Principios: **reproducibilidad** (código + datos + entorno + semilla versionados), **automatización** (pipelines, CI/CD/CT), **monitorización** y **gobernanza**.

Componentes y herramientas 2026:
| Componente | Función | Herramientas |
|---|---|---|
| Versionado de datos | *snapshots*, linaje | DVC, lakeFS, Delta/Iceberg *time travel* |
| Experiment tracking | parámetros, métricas, artefactos | **MLflow 3**, Weights & Biases, Neptune |
| Feature store | features consistentes offline/online, *point-in-time joins* | Feast, Tecton, Databricks/Vertex FS |
| Orquestación | DAGs de entrenamiento/inferencia | Airflow 3, Dagster, Prefect, Kubeflow, Metaflow |
| Registro de modelos | versiones, etapas, aprobaciones | MLflow Model Registry, Vertex/SageMaker Registry |
| Servicio | batch, online (REST/gRPC), streaming | FastAPI + Docker/K8s, BentoML, KServe, Triton, vLLM |
| Monitorización | deriva, calidad, rendimiento, coste | Evidently, WhyLabs, Arize, Grafana |

**Modos de despliegue** (Huyen DMLS Cap. 7): *batch* (predicciones precalculadas), *online* (síncrono, latencia crítica), *streaming* (features en tiempo real). Y el **compromiso** clave: modelo en la nube vs en el borde (latencia, privacidad, coste).

Implementamos un *tracker* de experimentos mínimo (para entender qué registra MLflow) y luego el equivalente con MLflow si está instalado.

In [ ]:
import hashlib, platform, sys, subprocess
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, log_loss

Xc, yc = make_classification(20000, 20, n_informative=8, weights=[0.85], random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(Xc, yc, test_size=0.3, random_state=0)

def huella_datos(X, y): return hashlib.sha256(np.ascontiguousarray(X).tobytes() + np.ascontiguousarray(y).tobytes()).hexdigest()[:12]
def git_sha():
    try: return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
    except Exception: return "n/a"

class TrackerMinimo:
    '''Lo esencial de un experiment tracker: quién/qué/cuándo/con qué datos/qué resultado, en un registro inmutable.'''
    def __init__(self, ruta): self.ruta = ruta
    def log_run(self, params, metrics, tags=None):
        run = {"run_id": hashlib.md5(json.dumps([params, time.time()]).encode()).hexdigest()[:8], "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
               "git": git_sha(), "python": sys.version.split()[0], "os": platform.platform(), "params": params, "metrics": metrics, "tags": tags or {}}
        with open(self.ruta, "a") as f: f.write(json.dumps(run) + "\n")
        return run["run_id"]
    def tabla(self): return pd.json_normalize([json.loads(l) for l in open(self.ruta)])

tracker = TrackerMinimo(os.path.join(tmp, "runs.jsonl"))
for lr_ in [0.03, 0.1, 0.3]:
    for leaves in [15, 63]:
        params = {"learning_rate": lr_, "max_leaf_nodes": leaves, "max_iter": 200, "data_hash": huella_datos(X_tr, y_tr)}
        m = HistGradientBoostingClassifier(learning_rate=lr_, max_leaf_nodes=leaves, max_iter=200, random_state=0).fit(X_tr, y_tr)
        p = m.predict_proba(X_te)[:, 1]
        tracker.log_run(params, {"roc_auc": round(roc_auc_score(y_te, p), 4), "log_loss": round(log_loss(y_te, p), 4)}, {"modelo": "HGB", "etapa": "exploracion"})
display(tracker.tabla()[["run_id", "params.learning_rate", "params.max_leaf_nodes", "metrics.roc_auc", "metrics.log_loss", "params.data_hash"]].sort_values("metrics.log_loss"))

In [ ]:
# OPCIONAL: lo mismo con MLflow (uv pip install mlflow). Añade UI (`mlflow ui`), artefactos, firma del modelo y registro.
try:
    import mlflow, mlflow.sklearn
    mlflow.set_tracking_uri(f"file:{os.path.join(tmp, 'mlruns')}"); mlflow.set_experiment("modulo9-demo")
    with mlflow.start_run(run_name="hgb-baseline") as run:
        m = HistGradientBoostingClassifier(learning_rate=0.1, max_leaf_nodes=31, random_state=0).fit(X_tr, y_tr)
        mlflow.log_params({"learning_rate": 0.1, "max_leaf_nodes": 31}); mlflow.log_metric("roc_auc", roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]))
        mlflow.sklearn.log_model(m, name="modelo", input_example=X_te[:2])
        print("MLflow run:", run.info.run_id, "| artefactos en", mlflow.get_artifact_uri())
except ImportError:
    print("MLflow no instalado; el tracker mínimo anterior ilustra el concepto.")

### Servir el modelo (OPCIONAL, para ejecutar fuera del notebook)
```python
# app.py — uv pip install fastapi uvicorn ; uvicorn app:app --port 8000
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np
app = FastAPI(); modelo = joblib.load("modelo.joblib")           # cargado una vez al arrancar
class Entrada(BaseModel): features: list[float]
@app.post("/predict")
def predict(e: Entrada):
    p = float(modelo.predict_proba(np.array([e.features]))[0, 1]); return {"prob": p, "version": "hgb-v3"}
```
Buenas prácticas: contenedor con dependencias fijadas (`uv lock`), *health checks*, latencia p95 medida, *shadow deployment* / *canary* (Huyen Cap. 9) antes del 100 % de tráfico, **registro de cada predicción con sus features** (imprescindible para la deriva).

## 4. Deriva de datos y monitorización (Huyen DMLS Cap. 8)

Tipos de cambio de distribución:
* **Covariate shift**: cambia $P(X)$, no $P(Y|X)$ (nuevos clientes más jóvenes).
* **Label shift**: cambia $P(Y)$ (más fraude en diciembre).
* **Concept drift**: cambia $P(Y|X)$ (la misma señal ya no predice lo mismo — el más peligroso y solo detectable con etiquetas).

Detección sin etiquetas (por feature): **PSI** (*Population Stability Index*, estándar en banca: <0.1 estable, 0.1-0.25 vigilar, >0.25 alerta), prueba **KS** para continuas, chi-cuadrado para categóricas; multivariante: **clasificador adversario** (entrenar a distinguir referencia vs actual: AUC≈0.5 ⇒ sin deriva). Implementamos las tres y las aplicamos a una ventana con deriva inyectada.

In [ ]:
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def psi(ref, act, bins=10):
    cortes = np.quantile(ref, np.linspace(0, 1, bins + 1)); cortes[0], cortes[-1] = -np.inf, np.inf
    pr = np.histogram(ref, cortes)[0] / len(ref) + 1e-6; pa = np.histogram(act, cortes)[0] / len(act) + 1e-6
    return float(np.sum((pa - pr) * np.log(pa / pr)))

X_ref = X_te.copy()
X_act = X_te.copy(); X_act[:, 0] += 0.8; X_act[:, 3] *= 1.6                     # deriva en dos features
X_act[:, 5] = np.where(rng.random(len(X_act)) < 0.3, X_act[:, 5] + 3, X_act[:, 5])  # deriva parcial (subpoblación)

filas = [{"feature": j, "PSI": psi(X_ref[:, j], X_act[:, j]), "KS_p": ks_2samp(X_ref[:, j], X_act[:, j]).pvalue} for j in range(X_ref.shape[1])]
rep = pd.DataFrame(filas); rep["alerta"] = np.select([rep.PSI > 0.25, rep.PSI > 0.1], ["ALERTA", "vigilar"], "ok")
display(rep.sort_values("PSI", ascending=False).head(6).round(4))

# detector multivariante: ¿puede un clasificador distinguir referencia de actual?
Z = np.r_[X_ref, X_act]; w = np.r_[np.zeros(len(X_ref)), np.ones(len(X_act))]
auc_adv = cross_val_score(LogisticRegression(max_iter=1000), Z, w, cv=5, scoring="roc_auc").mean()
print(f"AUC del clasificador adversario ref-vs-actual = {auc_adv:.3f}  (0.5 = sin deriva)")

# impacto real: rendimiento del modelo en la ventana con deriva (requiere etiquetas -> llega con retraso en producción)
m = HistGradientBoostingClassifier(random_state=0).fit(X_tr, y_tr)
print(f"ROC-AUC referencia = {roc_auc_score(y_te, m.predict_proba(X_ref)[:, 1]):.3f} | ventana con deriva = {roc_auc_score(y_te, m.predict_proba(X_act)[:, 1]):.3f}")

### Qué hacer cuando hay deriva (Huyen DMLS Cap. 9)
* **Reentrenar** con datos recientes: programado (semanal) o disparado por alerta; *continual learning* con evaluación **backtesting** antes de promover.
* **Pruebas en producción**: *shadow* (predice sin actuar), **A/B**, *canary* (1-5 % de tráfico), *interleaving*, **bandits** (asignación adaptativa).
* **Guardrails de datos** en la entrada del pipeline (Great Expectations, Pandera, esquemas Pydantic): un cambio de esquema o de unidad rompe silenciosamente más modelos que la deriva estadística.

## 5. LLMOps: lo que cambia con modelos fundacionales (Huyen AIE Cap. 9-10)

| Aspecto | ML clásico | Aplicaciones con LLM |
|---|---|---|
| Artefacto versionado | pesos + código | **prompts**, plantillas, índices RAG, configuraciones de herramientas (+ modelo del proveedor, que cambia sin avisar) |
| Evaluación | métricas con etiquetas | golden sets + jueces LLM + feedback de usuario (Módulo 6) |
| Coste dominante | entrenamiento | **inferencia** (tokens): caché, enrutamiento, cuantización, *batching* |
| Monitorización | deriva de features | deriva de *prompts* de usuarios, tasa de rechazo/alucinación, latencia por token, coste por sesión, inyecciones |
| Observabilidad | logs, métricas | **trazas** de cada paso del agente (OpenTelemetry GenAI semconv; Langfuse, LangSmith, Arize Phoenix) |

Modelo de coste simple: si un sistema RAG procesa 5 k tokens de entrada y 500 de salida por consulta con un modelo a 3 $/M entrada y 15 $/M salida → 0.0225 $/consulta; con 100 k consultas/día ≈ 2 250 $/día. *Prompt caching* (~90 % de descuento en la parte repetida) y un enrutador a un modelo pequeño para el 60 % de consultas simples bajan esto 3-5×. **Mide antes de optimizar.**

In [ ]:
def coste_diario(consultas, tok_in, tok_out, precio_in, precio_out, frac_cache=0.0, desc_cache=0.9, frac_router=0.0, ratio_barato=0.1):
    base = tok_in * precio_in / 1e6 + tok_out * precio_out / 1e6
    c = base * (1 - frac_cache * desc_cache * (tok_in * precio_in / 1e6) / base)      # el caché solo afecta a la parte de entrada
    c = c * ((1 - frac_router) + frac_router * ratio_barato)
    return consultas * c
esc = {"sin optimizar": dict(), "+ prompt caching 70%": dict(frac_cache=0.7), "+ router 60% a modelo barato": dict(frac_cache=0.7, frac_router=0.6)}
for k, kw in esc.items(): print(f"{k:32s} {coste_diario(100_000, 5000, 500, 3, 15, **kw):>10,.0f} $/día")

## 6. Ejercicios
1. **Polars/DuckDB.** Instala ambos y repite el benchmark con 20 M de filas. Añade una ventana temporal (media móvil de 30 días por cliente): compara la ergonomía del código.
2. **Point-in-time correctness.** Construye features agregadas por cliente y únelas a etiquetas fechadas *sin* usar información posterior a la fecha de la etiqueta (`pd.merge_asof`). Demuestra la fuga si lo haces mal.
3. **Escalado.** Con `torch.profiler`, mide el tiempo de un paso de entrenamiento del `MiniGPT` del Módulo 5 con lote 8 vs 64. Estima el *throughput* en tokens/s y el coste en GPU-horas de entrenar 1 B tokens.
4. **Pipeline reproducible.** Empaqueta el flujo de la sección 3 en un script con `argparse`, `uv lock`, semillas fijas y un `Makefile`/`justfile` con `make train evaluate`. Verifica que dos ejecuciones dan el mismo hash del modelo.
5. **Deriva realista.** Simula *concept drift* (cambia la relación $Y|X$ manteniendo $P(X)$). ¿Lo detectan PSI/KS/adversario? ¿Qué señal sí lo detecta y con qué retraso?
6. **Canary con bandits.** Implementa Thompson sampling para asignar tráfico entre el modelo actual y uno nuevo con tasas de acierto 0.80 y 0.83. ¿Cuántas muestras tarda en concentrar >90 % del tráfico en el mejor?

## Referencias
* Huyen, C. (2022). *Designing Machine Learning Systems*. O'Reilly. Cap. 3-10. Resumen: github.com/chiphuyen/dmls-book
* Huyen, C. (2025). *AI Engineering*. Cap. 9-10.
* Serra, J. (2024). *Deciphering Data Architectures*. O'Reilly. Cap. 1-3, 10-14.
* Géron, A. (2022). *Hands-On ML* (3.ª ed.). Cap. 19. · Zhang, A. et al. *D2L*. Cap. 13.
* Rajbhandari, S. et al. (2020). *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models*. SC. · Zhao, Y. et al. (2023). *PyTorch FSDP*. VLDB.
* Armbrust, M. et al. (2021). *Lakehouse: A New Generation of Open Platforms*. CIDR.
* Sculley, D. et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
* Kreuzberger, D. et al. (2023). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access.
* Docs: PyTorch Distributed · Hugging Face Accelerate · Ray Train · MLflow · Polars · DuckDB · Apache Spark 4 · Delta Lake / Apache Iceberg.